# Predictive Maintenance — Model Training & Tuning

Trains 3 models (Random Forest, XGBoost, LightGBM), tunes each with `RandomizedSearchCV`, picks the best, and saves everything needed for the Streamlit app.

In [1]:
import pandas as pd
import numpy as np
import joblib, json, warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

RANDOM_STATE = 42

## 1. Load data

In [2]:
df = pd.read_csv("industrial_predictive_maintenance_dataset.csv")
print(df.shape)
df.head()

(20000, 18)


,Machine_ID,Vibration,Temperature,Smoke_Level,Current,Voltage,RPM,Pressure,Humidity,Acoustic_Level,Power_Consumption,Operating_Hours,Load_Percentage,Maintenance_History,Days_Since_Maintenance,Machine_Age,Failure_Risk_Score,Machine_Status
0,36,2.80,45.48,8.45,48.72,229.1,1695,6.68,49.8,63.4,11.166,2168,42.9,2,487,2,11.84,Normal
1,44,2.31,54.56,7.54,31.85,227.0,1700,5.41,46.7,64.6,7.166,9274,42.6,25,276,9,9.94,Normal
2,71,4.65,57.27,21.41,34.78,222.9,1476,6.53,48.1,68.4,7.520,14604,17.8,50,632,20,19.46,Normal
3,38,5.10,63.60,13.32,42.39,244.5,1341,5.04,33.8,69.0,10.240,18224,67.9,23,278,21,15.96,Normal
4,36,2.45,49.57,4.62,43.45,222.9,1819,6.23,82.5,63.9,9.908,1519,47.1,2,529,2,1.73,Normal


## 2. Train/test split
Target is label-encoded; features are used as-is (tree models don't need scaling).

In [3]:
X = df.drop(columns=["Machine_Status"])
y_raw = df["Machine_Status"]

le = LabelEncoder()
y = le.fit_transform(y_raw)
print(dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape

{'Critical': np.int64(0), 'Failure': np.int64(1), 'Normal': np.int64(2), 'Warning': np.int64(3)}


((16000, 17), (4000, 17))

## 3. Hyperparameter tuning (3 models)
Each model gets a `RandomizedSearchCV` (3-fold CV, macro-F1 scoring) on a stratified subsample for speed, then the best params are refit on the **full** training set.

In [4]:
param_grids = {
    "RandomForest": {
        "n_estimators": [150, 250, 350],
        "max_depth": [8, 12, 16, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
    },
    "XGBoost": {
        "n_estimators": [150, 250, 350],
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.03, 0.05, 0.1, 0.2],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
    },
    "LightGBM": {
        "n_estimators": [150, 250, 350],
        "num_leaves": [15, 31, 63],
        "learning_rate": [0.03, 0.05, 0.1, 0.2],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
    },
}

base_models = {
    "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost": XGBClassifier(random_state=RANDOM_STATE, eval_metric="mlogloss", n_jobs=-1),
    "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}

# Search on a stratified subsample (faster tuning), then refit best params on full data
_, X_search, _, y_search = train_test_split(
    X_train, y_train, test_size=0.35, random_state=RANDOM_STATE, stratify=y_train
)

In [ ]:
tuned_models = {}
search_results = {}

for name, model in base_models.items():
    print(f"Tuning {name} ...", flush=True)
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grids[name],
        n_iter=8,
        cv=3,
        scoring="f1_macro",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )
    search.fit(X_search, y_search)
    print(f"  best CV f1_macro (subsample): {search.best_score_:.4f}", flush=True)
    print(f"  best params: {search.best_params_}", flush=True)

    # Refit the winning config on the full training set
    final_model = search.best_estimator_.__class__(**search.best_estimator_.get_params())
    final_model.fit(X_train, y_train)

    tuned_models[name] = final_model
    search_results[name] = search.best_score_
    print(f"  refit on full training set done\n", flush=True)

Tuning RandomForest ...
  best CV f1_macro (subsample): 0.8357
  best params: {'n_estimators': 350, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': None}
  refit on full training set done

Tuning XGBoost ...
  best CV f1_macro (subsample): 0.8362
  best params: {'subsample': 0.85, 'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.03, 'colsample_bytree': 0.85}
  refit on full training set done

Tuning LightGBM ...


  best CV f1_macro (subsample): 0.8357


  best params: {'n_estimators': 350, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': None}


  refit on full training set done



Tuning XGBoost ...


...

  refit on full training set done



Tuning LightGBM ...


  best CV f1_macro (subsample): 0.8299


  best params: {'subsample': 0.7, 'num_leaves': 15, 'n_estimators': 250, 'learning_rate': 0.03, 'colsample_bytree': 0.7}


  refit on full training set done



## 4. Compare tuned models on the held-out test set

In [6]:
results = []
for name, model in tuned_models.items():
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    f1m = f1_score(y_test, pred, average="macro")
    results.append({"model": name, "cv_f1_macro": search_results[name], "test_accuracy": acc, "test_f1_macro": f1m})

results_df = pd.DataFrame(results).sort_values("test_f1_macro", ascending=False).reset_index(drop=True)
results_df

,model,cv_f1_macro,test_accuracy,test_f1_macro
0,XGBoost,0.836234,0.90150,0.836463
1,LightGBM,0.829916,0.89900,0.831724
2,RandomForest,0.835684,0.90025,0.831207


## 5. Pick the best model

In [7]:
best_name = results_df.iloc[0]["model"]
best_model = tuned_models[best_name]
print("Best model:", best_name)

pred = best_model.predict(X_test)
print(classification_report(y_test, pred, target_names=le.classes_))
print("Confusion matrix:")
print(confusion_matrix(y_test, pred))

Best model: XGBoost
              precision    recall  f1-score   support

    Critical       0.68      0.69      0.69       400
     Failure       0.83      0.85      0.84       400
      Normal       0.97      0.97      0.97      2300
     Warning       0.87      0.84      0.85       900

    accuracy                           0.90      4000
   macro avg       0.83      0.84      0.84      4000
weighted avg       0.90      0.90      0.90      4000

Confusion matrix:
[[ 278   71    0   51]
 [  60  340    0    0]
 [   0    0 2235   65]
 [  72    0   75  753]]


## 6. Feature importance (best model)

In [8]:
importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances

Smoke_Level               0.680002
Failure_Risk_Score        0.138805
Vibration                 0.041455
Temperature               0.015497
Acoustic_Level            0.014220
Machine_ID                0.011039
Load_Percentage           0.010985
Operating_Hours           0.010971
Maintenance_History       0.010796
Current                   0.008941
Pressure                  0.008882
Humidity                  0.008810
Days_Since_Maintenance    0.008699
Power_Consumption         0.008520
Machine_Age               0.007616
RPM                       0.007555
Voltage                   0.007206
dtype: float32

## 7. Save artifacts for Streamlit
Everything the app needs to load and run predictions: the trained model, the label encoder, and the exact feature order.

In [9]:
joblib.dump(best_model, "best_model.pkl")
joblib.dump(le, "label_encoder.pkl")

with open("feature_columns.json", "w") as f:
    json.dump(list(X.columns), f)

with open("model_info.json", "w") as f:
    json.dump({
        "best_model": best_name,
        "test_accuracy": float(results_df.iloc[0]["test_accuracy"]),
        "test_f1_macro": float(results_df.iloc[0]["test_f1_macro"]),
        "classes": list(le.classes_),
    }, f, indent=2)

print("Saved: best_model.pkl, label_encoder.pkl, feature_columns.json, model_info.json")

Saved: best_model.pkl, label_encoder.pkl, feature_columns.json, model_info.json


**Next step:** in the Streamlit app, load `best_model.pkl` + `label_encoder.pkl`, build the input form using the columns in `feature_columns.json`, predict, then `label_encoder.inverse_transform(...)` to show the class name.